In [49]:
#--------------------------------------------------------------------Weekly Assignment-2-------------------------------------------------------------------------------

In [50]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
nltk.download('stopwords')
nltk.download('wordnet')
nltk.download('omw-1.4')
nltk.download('averaged_perceptron_tagger_eng')

from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords, wordnet
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk import pos_tag

import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.cluster import KMeans
from sklearn.model_selection import train_test_split
from sklearn.datasets import fetch_20newsgroups
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, silhouette_score)
from sklearn.decomposition import TruncatedSVD
import matplotlib.pyplot as plt

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


In [51]:
# Step 1 - Input Text
text = """
Climate change, often referred to as global warming, is one of the most pressing challenges
facing humanity today: rising temperatures, extreme weather events, and melting polar ice!
Scientists across the globe are asking — can we still reverse the damage done to our planet?
While some policymakers believe it\'s possible; others remain deeply pessimistic. The major
drivers of climate change include: greenhouse gas emissions, deforestation, and industrial
pollution (CO2 and methane). As of 2025, global temperatures have risen by approximately 1.2°C
above pre-industrial levels — triggering floods, droughts, and wildfires worldwide!

Renewable energy is considered the most promising solution to combat climate change; it
encompasses technologies such as solar panels, wind turbines, and hydroelectric power. Sectors
like: transportation, manufacturing, and agriculture must urgently transition away from fossil
fuels! However, obstacles remain — cost, infrastructure limitations, and political resistance
make widespread adoption challenging. For instance, the statement "clean energy is too expensive"
is increasingly false; solar energy costs have dropped by over 90% in the last decade.
Governments, businesses, and individuals must collaborate to accelerate this transition,
investing in innovation and policy reform every year.
"""

print("=== Original Text ===")
print(text)

=== Original Text ===

Climate change, often referred to as global warming, is one of the most pressing challenges
facing humanity today: rising temperatures, extreme weather events, and melting polar ice!
Scientists across the globe are asking — can we still reverse the damage done to our planet?
While some policymakers believe it's possible; others remain deeply pessimistic. The major
drivers of climate change include: greenhouse gas emissions, deforestation, and industrial
pollution (CO2 and methane). As of 2025, global temperatures have risen by approximately 1.2°C
above pre-industrial levels — triggering floods, droughts, and wildfires worldwide!

Renewable energy is considered the most promising solution to combat climate change; it
encompasses technologies such as solar panels, wind turbines, and hydroelectric power. Sectors
like: transportation, manufacturing, and agriculture must urgently transition away from fossil
fuels! However, obstacles remain — cost, infrastructure limit

In [52]:
#Step 2- Tokenization
tokens = word_tokenize(text)
print("Total tokens:", len(tokens))

#Step 3 - Remove Punctuation
tokens_no_punct = [token for token in tokens if token.isalpha()]
print("Tokens after punctuation removal:", len(tokens_no_punct))

#Step 4 - Stop word removal
stop_words = set(stopwords.words('english'))
tokens_no_stop = [token for token in tokens_no_punct if token.lower() not in stop_words]
print("Tokens after stop word removal:", len(tokens_no_stop))

#Step 5 - Stemming(porter stemmer)
stemmer = PorterStemmer()
stemmed_tokens = [stemmer.stem(token) for token in tokens_no_stop]
print("Stemmed tokens:", len(stemmed_tokens))

#Step 6 - Lemmatization
lemmatizer = WordNetLemmatizer()
lemmatized_tokens = [lemmatizer.lemmatize(token.lower()) for token in tokens_no_stop]
print("Lemmatized tokens:", len(lemmatized_tokens))

#Step 7 - Lemmatization with POS Tagging
lemmatizer1 = WordNetLemmatizer()

def get_pos(word):
    tag = pos_tag([word])[0][1][0].upper()
    tag_dict = {"J": wordnet.ADJ,
                "N": wordnet.NOUN,
                "V": wordnet.VERB,
                "R": wordnet.ADV}
    return tag_dict.get(tag, wordnet.NOUN)

lemmatized_tokens1 = [lemmatizer1.lemmatize(token, get_pos(token)) for token in tokens_no_stop]
print("POS-Lemmatized tokens:", len(lemmatized_tokens1))



Total tokens: 226
Tokens after punctuation removal: 176
Tokens after stop word removal: 123
Stemmed tokens: 123
Lemmatized tokens: 123
POS-Lemmatized tokens: 123


In [53]:
#Display preprocessed Text
preprocessed_text = " ".join(lemmatized_tokens1)
print("Preprocessed text:")
print(preprocessed_text)

Preprocessed text:
Climate change often refer global warm one press challenge face humanity today rise temperature extreme weather event melt polar ice Scientists across globe ask still reverse damage do planet policymakers believe possible others remain deeply pessimistic major driver climate change include greenhouse gas emission deforestation industrial pollution methane global temperature risen approximately level trigger flood drought wildfire worldwide Renewable energy consider promising solution combat climate change encompasses technology solar panel wind turbine hydroelectric power Sectors like transportation manufacturing agriculture must urgently transition away fossil fuel However obstacle remain cost infrastructure limitation political resistance make widespread adoption challenge instance statement clean energy expensive increasingly false solar energy cost drop last decade Governments business individual must collaborate accelerate transition invest innovation policy ref

In [54]:
# Feature Engineering
# Bag of Words
bow_vectorizer = CountVectorizer()
bow_matrix = bow_vectorizer.fit_transform([preprocessed_text])
print("BoW Vector:", bow_matrix.toarray())

BoW Vector: [[1 1 1 1 1 1 1 1 1 2 3 1 3 1 1 1 2 1 1 1 1 1 1 1 1 1 1 3 1 1 1 1 1 1 1 1
  1 1 2 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 1
  1 1 1 1 1 1 1 1 1 1 2 1 1 1 1 1 1 1 2 1 1 1 1 2 1 2 1 1 1 1 1 1 1 1 1 1
  1]]


In [55]:
# TF-IDF
tfidf_vectorizer = TfidfVectorizer()
tfidf_matrix = tfidf_vectorizer.fit_transform([preprocessed_text])
print("TF-IDF Vector:", tfidf_matrix.toarray())

TF-IDF Vector: [[0.07980869 0.07980869 0.07980869 0.07980869 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.15961738 0.23942607 0.07980869
  0.23942607 0.07980869 0.07980869 0.07980869 0.15961738 0.07980869
  0.07980869 0.07980869 0.07980869 0.07980869 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.23942607 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.07980869 0.07980869 0.07980869
  0.07980869 0.07980869 0.15961738 0.07980869 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.07980869 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.07980869 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.07980869 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.15961738 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.07980869 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.07980869 0.07980869 0.07980869
  0.07980869 0.07980869 0.07980869 0.07980869 0.15961738 0.07980869
  0.07980869 0.07980869 0.0798086

In [56]:
#SUPERVISED - Logistic  Regression (Text classification)
#Load data
categories = ['sci.med', 'sci.space', 'talk.politics.guns', 'comp.graphics']
train_data = fetch_20newsgroups(subset='train', categories=categories, random_state=42)
test_data  = fetch_20newsgroups(subset='test',  categories=categories, random_state=42)

print("Train:", len(train_data.data))
print("Test:", len(test_data.data))

Train: 2317
Test: 1543


In [57]:
# TF-IDF + Logistic Regression
tfidf_vec = TfidfVectorizer(stop_words='english')
X_train = tfidf_vec.fit_transform(train_data.data)
X_test  = tfidf_vec.transform(test_data.data)

model = LogisticRegression(max_iter=1000)
model.fit(X_train, train_data.target)

y_pred = model.predict(X_test)
print("Accuracy:", accuracy_score(test_data.target, y_pred))
print()
print(classification_report(test_data.target, y_pred, target_names=categories))


Accuracy: 0.9377835385612443

                    precision    recall  f1-score   support

           sci.med       0.87      0.97      0.92       389
         sci.space       0.93      0.91      0.92       396
talk.politics.guns       0.98      0.92      0.95       394
     comp.graphics       0.98      0.95      0.97       364

          accuracy                           0.94      1543
         macro avg       0.94      0.94      0.94      1543
      weighted avg       0.94      0.94      0.94      1543



In [58]:
#UNPERVISED - K-Means clustering
#K-Means on the same TF-IDF data (labels ignored)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
clusters = kmeans.fit_predict(X_train)

# Cluster sizes
for i in range(4):
    count = 0
    for label in clusters:
        if label == i:
            count = count + 1
    print("Cluster", i, ":", count, "docs")

# Top 5 words per cluster
feature_names = tfidf_vec.get_feature_names_out()
centers = kmeans.cluster_centers_
print()
for i in range(4):
    sorted_indices = np.argsort(centers[i])
    top_indices = sorted_indices[-5:]
    top_indices = top_indices[::-1]
    print("Cluster", i, "top words:", feature_names[top_indices])


Cluster 0 : 402 docs
Cluster 1 : 79 docs
Cluster 2 : 1391 docs
Cluster 3 : 445 docs

Cluster 0 top words: ['space' 'nasa' 'henry' 'gov' 'edu']
Cluster 1 top words: ['geb' 'pitt' 'banks' 'gordon' 'cs']
Cluster 2 top words: ['edu' 'com' 'subject' 'lines' 'organization']
Cluster 3 top words: ['gun' 'edu' 'com' 'guns' 'people']
